# KALIA — Training (GPU T4 x2)

**Notebook settings**
- Accelerator: **GPU T4 x2**
- Internet: **On**
- Add-ons → Secrets: add `HF_TOKEN` (a HuggingFace *write* token)
- Add the **kalia-tokens** dataset to this notebook

Checkpoints sync to your private HF repo every 30 minutes, so sessions can be interrupted safely. Re-running the last cell resumes exactly where it stopped.

In [ ]:
!pip install -q tiktoken pyyaml huggingface_hub

In [ ]:
import glob
import os
import shutil
import subprocess

work = "/kaggle/working/kalia"
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/**/prepare.py", recursive=True))
    if hits:
        shutil.copytree(os.path.dirname(hits[0]), work)
        print("code copied from", os.path.dirname(hits[0]))
    else:
        from kaggle_secrets import UserSecretsClient

        token = ""
        try:
            token = UserSecretsClient().get_secret("GH_TOKEN")
        except Exception:
            pass
        url = "https://github.com/subhajitlucky/kalia.git"
        if token:
            url = url.replace("https://", f"https://x-access-token:{token}@")
        r = subprocess.run(["git", "clone", "--quiet", url, work], capture_output=True, text=True)
        assert r.returncode == 0, "clone failed - attach the kalia-code dataset or GH_TOKEN secret"
os.chdir(work)
if not os.path.exists("kalia-m.yaml"):
    cands = glob.glob("/kaggle/input/**/kalia-m.yaml", recursive=True)
    if cands:
        shutil.copy(cands[0], "kalia-m.yaml")
assert os.path.exists("kalia-m.yaml"), "kalia-m.yaml not found in attached inputs"
print("working in", os.getcwd())

In [ ]:
import glob
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

hits = glob.glob("/kaggle/input/**/train.bin", recursive=True)
assert hits, "no train.bin found - attach the kalia-tokens dataset or the kalia-prep kernel output"
DATA_DIR = str(Path(hits[0]).parent)
out_dir = "/kaggle/working/out"
HUB_REPO = "kalia-lm/kalia-v012"  # <-- HuggingFace repo: (org or user)/name
print("DATA_DIR =", DATA_DIR)

from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(HUB_REPO, repo_type="model", private=True, exist_ok=True)
test_path = "/kaggle/working/hf_connection_test.txt"
with open(test_path, "w") as fh:
    fh.write("KALIA connection test OK")
api.upload_file(path_or_fileobj=test_path, path_in_repo="logs/connection_test.txt", repo_id=HUB_REPO)
print("HF connection test passed ->", HUB_REPO)

In [ ]:
import os
import subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "torchrun", "--nproc_per_node=2", "--standalone", "train.py",
    "--config", "kalia-m.yaml",
    "--data-dir", DATA_DIR,
    "--out-dir", out_dir,
    "--resume",
    "--hub-repo", HUB_REPO,
    "--max-minutes", "510",
]
result = subprocess.run(cmd)
assert result.returncode == 0, f"training exited with code {result.returncode}"

**Notes**
- If torchrun complains about 2 GPUs, change `--nproc_per_node=2` to `1`.
- First run: no checkpoint exists yet, so it starts from random weights.
- Later sessions: automatically resumes from the latest checkpoint on HF Hub.
- Progress lives in your HF repo at `logs/train_log.csv`.